# Lesson 5.3 — Guardrail Layering, Least Privilege, and Safety Hooks

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Layer input, model, and output guardrails (defense in depth)
- Apply least-privilege and secure-by-design principles
- Use hooks to block destructive actions deterministically
- Combine guardrails into a single request pipeline

**You will produce:** A layered guardrail pipeline with input/output filters and a destructive-action hook.

## Concepts

**Guardrail layering** (defense in depth) places independent checks at multiple stages:

- **Input guardrails**: sanitize/validate user input, detect injection and PII (Lessons 5.1-5.2).
- **Model/tool guardrails**: least-privilege tool sets, approval gates, and **hooks** that block destructive actions deterministically (Phase 4).
- **Output guardrails**: validate structured output, filter disallowed content, and re-check for leaked secrets before returning to the user.

**Secure-by-design** principles: least privilege (grant the minimum access needed), fail closed (deny on uncertainty), and content-policy enforcement. No single layer is sufficient; combine them so a bypass of one is caught by another.

> **Reference:** [Anthropic - Strengthen guardrails](https://docs.anthropic.com/en/docs/test-and-evaluate/strengthen-guardrails/reduce-hallucinations)

In [ ]:
# !pip install anthropic
import os

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {prompt[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Input and output filters

Two independent checks around the model call.

In [ ]:
BANNED = {"malware", "bioweapon"}

def input_guard(text):
    low = text.lower()
    if any(b in low for b in BANNED):
        return False, "blocked by input policy"
    return True, text

def output_guard(text):
    # fail closed: redact anything that looks like a key
    import re
    return re.sub(r"sk-[A-Za-z0-9]{8,}", "[REDACTED_KEY]", text)

ok, val = input_guard("How do I write malware?")
print(ok, val)
print(output_guard("your key is sk-ABCD12345678"))

## Code-Along: Step 2 — Destructive-action hook

A deterministic gate that fails closed on sensitive tools.

In [ ]:
DESTRUCTIVE = {"delete_file", "drop_table", "wipe"}

def tool_hook(name, inp, approved=False):
    if name in DESTRUCTIVE and not approved:
        return {"allowed": False, "reason": "destructive action requires approval"}
    return {"allowed": True}

print(tool_hook("drop_table", {"t": "users"}))
print(tool_hook("drop_table", {"t": "users"}, approved=True))

## Code-Along: Step 3 — Compose the pipeline

Chain input guard -> model call -> output guard.

In [ ]:
def guarded_request(user_text):
    ok, val = input_guard(user_text)
    if not ok:
        return val
    raw = mock_text(val) if not HAVE_API else client.messages.create(
        model=MODEL_HAIKU, max_tokens=150,
        messages=[{"role": "user", "content": val}]).content[0].text
    return output_guard(raw)

print(guarded_request("Summarize our refund policy."))

## Exercise

Add a third guardrail layer: a `rate_limit(user_id)` function that blocks a user after N requests in the current session, and wire it into `guarded_request`.

In [ ]:
_counts = {}
def rate_limit(user_id, limit=5):
    # TODO: increment _counts[user_id]; return False if over limit
    pass

# TODO: call rate_limit at the start of guarded_request and block if exceeded

## Key Takeaways

- Layer guardrails at input, model/tool, and output stages (defense in depth).
- Apply least privilege and fail closed on uncertainty.
- Use hooks to block destructive actions deterministically.
- No single layer suffices - combine independent checks.

---

**Next:** [Lesson 5.4 — Secrets, Keys, and Identity Management](./lesson-5.4-secrets-and-key-management.ipynb)